# S30 - EA2 Unidad 1 - Modelos de Predicción y Clasificación

Notebook de **evidencias**. Usa las mismas clases y la misma configuración (`src/config.py`)
que `main.py`, el script que ejecuta GitHub Actions.

```
Hugging Face → DataFrame → preprocesamiento
                         → regresión lineal + evaluación
                         → clasificación + evaluación
                         → resumen automático .txt
```

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.pipeline import load_raw_data, build_preprocessor, run_experiment
from src.preprocessing import split_data
from src.regression_model import LinearRegressionModel
from src.classification_model import LogisticRegressionClassifier
from src.interpretation import (interpret_regression, interpret_classification,
                                describe_class_balance)
from src.report_writer import ModelSummaryWriter

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print("Semilla:", config.RANDOM_STATE, "| test_size:", config.TEST_SIZE)

Semilla: 42 | test_size: 0.3


## 0. Construcción automática del DataFrame

| Campo | Valor |
|---|---|
| Identificador | `scikit-learn/iris` |
| Enlace | https://huggingface.co/datasets/scikit-learn/iris |
| Split | `train` (único disponible) |
| Contenido | 150 flores de 3 especies de iris con 4 medidas en cm |

Columnas relevantes: `SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm` (numéricas
continuas) y `Species` (categórica). `Id` es solo un consecutivo.

- **Regresión:** objetivo numérico continuo → `petal_length`.
- **Clasificación:** objetivo categórico → `species`. Es una etiqueta original del dataset,
  **no** se construyó a partir de una variable numérica, así que no hay variable "fuente" que excluir.

Los datos se descargan con `datasets.load_dataset()` dentro de `HFDatasetLoader`; no hay archivos manuales.

In [2]:
df_raw = load_raw_data()

print("Dataset:", config.DATASET_ID, "| split:", config.DATASET_SPLIT)
print("Tipo:", type(df_raw).__name__)
print("Tamaño (filas, columnas):", df_raw.shape)
print("Columnas:", list(df_raw.columns))

Dataset: scikit-learn/iris | split: train
Tipo: DataFrame
Tamaño (filas, columnas): (150, 6)
Columnas: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


In [3]:
df_raw.dtypes

Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species              str
dtype: object

In [4]:
df_raw.head(8)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
5,6,5.4,3.9,1.7,0.4,Iris-setosa
6,7,4.6,3.4,1.4,0.3,Iris-setosa
7,8,5.0,3.4,1.5,0.2,Iris-setosa


## 1. Preprocesamiento

### 1.1 Estado inicial
Se revisan valores nulos, duplicados (ignorando `Id`, porque con él cada fila es única
artificialmente) y la forma de las etiquetas.

In [5]:
print("Valores nulos por columna:")
print(df_raw.isnull().sum())
print()
print("Duplicados contando Id:   ", int(df_raw.duplicated().sum()))
print("Duplicados sin contar Id: ", int(df_raw.drop(columns="Id").duplicated().sum()))

Valores nulos por columna:
Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

Duplicados contando Id:    0
Duplicados sin contar Id:  3


In [6]:
# Filas que se repiten en todas sus medidas y especie
measures = df_raw.drop(columns="Id")
df_raw[measures.duplicated(keep=False)].sort_values(list(measures.columns))

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
9,10,4.9,3.1,1.5,0.1,Iris-setosa
34,35,4.9,3.1,1.5,0.1,Iris-setosa
37,38,4.9,3.1,1.5,0.1,Iris-setosa
101,102,5.8,2.7,5.1,1.9,Iris-virginica
142,143,5.8,2.7,5.1,1.9,Iris-virginica


In [7]:
df_raw["Species"].value_counts()

Species
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64

### 1.2 Limpieza (`DataPreprocessor.clean`)

Estas operaciones **no aprenden nada de los datos**, así que pueden aplicarse antes de la división:

1. **Renombrar columnas**: nombres en snake_case.
2. **Eliminar `id`**: no describe la flor. Además, el archivo está ordenado por especie, así que
   `id` "delataría" la clase (fuga de información).
3. **Normalizar etiquetas y tipo**: `Iris-setosa` → `setosa`, tipo `category`.
4. **Nulos**: se revisan (no hay) y se eliminarían si existieran.
5. **Duplicados**: 3 flores aparecen dos veces. Se eliminan para que una misma flor no quede en
   entrenamiento y en prueba a la vez, lo que inflaría las métricas.

In [8]:
preprocessor = build_preprocessor()
df_clean = preprocessor.clean(df_raw)
preprocessor.steps_as_dataframe()

,operation,reason,effect
0,Renombrar columnas,"Nombres en snake_case, sin la unidad 'Cm'.","['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species'] -> ['id', 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']"
1,Eliminar columnas,"'id' solo numera las filas y el archivo está ordenado por especie, así que delataría la clase.",Columnas eliminadas: ['id']; quedan 5.
2,Normalizar etiquetas y corregir tipo,El prefijo 'Iris-' es redundante y la especie es categórica.,"['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'] -> ['setosa', 'versicolor', 'virginica'] (tipo 'category')."
3,Revisar valores nulos,scikit-learn no acepta valores faltantes.,Nulos encontrados: 0; filas eliminadas: 0.
4,Eliminar filas duplicadas,Una fila repetida en train y test a la vez infla las métricas.,Duplicados encontrados: 3; filas: 150 -> 147.


In [9]:
comparison = pd.DataFrame({
    "antes": [df_raw.shape[0], df_raw.shape[1], int(df_raw.drop(columns="Id").duplicated().sum()),
              int(df_raw.isnull().sum().sum()), str(df_raw["Species"].dtype),
              ", ".join(sorted(df_raw["Species"].unique()))],
    "después": [df_clean.shape[0], df_clean.shape[1], int(df_clean.duplicated().sum()),
                int(df_clean.isnull().sum().sum()), str(df_clean["species"].dtype),
                ", ".join(df_clean["species"].cat.categories)],
}, index=["filas", "columnas", "duplicados", "nulos", "tipo de la especie", "etiquetas"])
comparison

,antes,después
filas,150,147
columnas,6,5
duplicados,3,0
nulos,0,0
tipo de la especie,str,category
etiquetas,"Iris-setosa, Iris-versicolor, Iris-virginica","setosa, versicolor, virginica"


In [10]:
df_clean.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [11]:
df_clean.dtypes

sepal_length     float64
sepal_width      float64
petal_length     float64
petal_width      float64
species         category
dtype: object

### 1.3 Relación entre variables
La correlación ayuda a elegir predictores y anticipa la **multicolinealidad** (variables muy
correlacionadas entre sí), que afecta la interpretación de los coeficientes de la regresión.

In [12]:
df_clean.select_dtypes("number").corr().round(3)

,sepal_length,sepal_width,petal_length,petal_width
sepal_length,1.000,-0.109,0.871,0.817
sepal_width,-0.109,1.000,-0.421,-0.356
petal_length,0.871,-0.421,1.000,0.962
petal_width,0.817,-0.356,0.962,1.000


### 1.4 Transformaciones que aprenden de los datos (sin fuga de información)

El **escalado** (`StandardScaler`) y la **codificación** (`OneHotEncoder`) aprenden parámetros
(medias, desviaciones, categorías). Por eso van **dentro de un `Pipeline`** que se ajusta solo con
`X_train` y después se aplica a `X_test`.

- División: 70 % entrenamiento / 30 % prueba, `random_state=42`, estratificada por especie.
- Estratificar mantiene la misma proporción de especies en ambos conjuntos.

## 2. Modelo de regresión lineal

**Pregunta:** ¿se puede estimar el **largo del pétalo** (cm) de una flor a partir de sus otras
medidas y su especie?

- **Objetivo:** `petal_length` (numérica continua, 1.0 a 6.9 cm).
- **Predictoras:** `sepal_length`, `sepal_width`, `petal_width` y `species` (codificada one-hot).
  Las medidas numéricas tienen una relación aproximadamente lineal con el largo del pétalo
  (ver correlaciones), y la especie aporta el tamaño típico de cada grupo.
- **Algoritmo:** `LinearRegression` (mínimos cuadrados ordinarios) con **parámetros
  predeterminados** de scikit-learn.
- **Preprocesamiento en el Pipeline:** `StandardScaler` en las numéricas y
  `OneHotEncoder(drop="first")` en `species`. `setosa` queda como categoría de referencia.

In [13]:
regression = LinearRegressionModel(config.REGRESSION_NUMERIC_FEATURES,
                                   config.REGRESSION_CATEGORICAL_FEATURES,
                                   config.REGRESSION_TARGET)
Xr_train, Xr_test, yr_train, yr_test = split_data(
    df_clean, regression.features, config.REGRESSION_TARGET,
    config.TEST_SIZE, config.RANDOM_STATE, stratify_column=config.LABEL_COLUMN)

regression.fit(Xr_train, yr_train)

print("Entrenamiento:", Xr_train.shape, "| Prueba:", Xr_test.shape)
print("Parámetros de LinearRegression:", regression.params())

Entrenamiento: (102, 4) | Prueba: (45, 4)
Parámetros de LinearRegression: {'copy_X': True, 'fit_intercept': True, 'n_jobs': None, 'positive': False, 'tol': 1e-06}


**Evidencia de que el escalado se ajustó solo con entrenamiento:** las medias que guardó el
`StandardScaler` coinciden con las de `X_train`. Tras transformar, entrenamiento queda con media 0
y desviación 1, mientras que prueba queda *cerca* de 0 pero no exactamente, porque se le aplicaron
los parámetros de entrenamiento.

In [14]:
preprocess = regression.pipeline.named_steps["preprocess"]
scaler = preprocess.named_transformers_["num"]
num = config.REGRESSION_NUMERIC_FEATURES

print("¿Medias del scaler == medias de X_train?",
      np.allclose(scaler.mean_, Xr_train[num].mean()))

names = preprocess.get_feature_names_out()
train_t = pd.DataFrame(preprocess.transform(Xr_train), columns=names)
test_t = pd.DataFrame(preprocess.transform(Xr_test), columns=names)

pd.DataFrame({
    "media train antes": Xr_train[num].mean(),
    "media train después": train_t[num].mean(),
    "desv. train después": train_t[num].std(ddof=0),
    "media test después": test_t[num].mean(),
}).round(3)

¿Medias del scaler == medias de X_train? True


,media train antes,media train después,desv. train después,media test después
sepal_length,5.878,-0.0,1.0,-0.082
sepal_width,3.040,-0.0,1.0,0.112
petal_width,1.218,-0.0,1.0,-0.037


In [15]:
# Codificación one-hot de la especie (antes → después)
pd.concat([Xr_train[["species"]].reset_index(drop=True),
           train_t[["species_versicolor", "species_virginica"]]], axis=1).drop_duplicates()

,species,species_versicolor,species_virginica
0,setosa,0.0,0.0
6,virginica,0.0,1.0
7,versicolor,1.0,0.0


### 2.1 Evaluación en prueba

In [16]:
reg_metrics = regression.evaluate(Xr_test, yr_test)
reg_cv = regression.cross_validate(Xr_train, yr_train, config.CV_FOLDS)

pd.Series({
    "MAE (cm)": reg_metrics["mae"],
    "RMSE (cm)": reg_metrics["rmse"],
    "R²": reg_metrics["r2"],
    "MAE modelo base (media)": reg_metrics["baseline_mae"],
    f"R² CV {config.CV_FOLDS} pliegues (media)": reg_cv["r2_mean"],
    f"R² CV {config.CV_FOLDS} pliegues (desv.)": reg_cv["r2_std"],
}).round(4)

MAE (cm)                    0.2199
RMSE (cm)                   0.2921
R²                          0.9705
MAE modelo base (media)     1.5159
R² CV 5 pliegues (media)    0.9748
R² CV 5 pliegues (desv.)    0.0131
dtype: float64

In [17]:
predictions = Xr_test.copy()
predictions["real"] = yr_test
predictions["predicho"] = regression.predict(Xr_test).round(2)
predictions["error_abs"] = (predictions["real"] - predictions["predicho"]).abs().round(2)
predictions.sort_values("error_abs", ascending=False).head(10)

,sepal_length,sepal_width,petal_width,species,real,predicho,error_abs
139,6.9,3.1,2.3,virginica,5.1,5.90,0.80
132,6.1,2.6,1.4,virginica,5.6,4.90,0.70
14,5.8,4.0,0.2,setosa,1.2,1.87,0.67
96,5.1,2.5,1.1,versicolor,3.0,3.62,0.62
82,5.4,3.0,1.5,versicolor,4.5,3.98,0.52
35,5.5,3.5,0.2,setosa,1.3,1.77,0.47
105,7.3,2.9,1.8,virginica,6.3,5.87,0.43
64,5.6,3.0,1.5,versicolor,4.5,4.11,0.39
61,6.1,2.9,1.4,versicolor,4.7,4.38,0.32
42,5.1,3.8,0.4,setosa,1.9,1.59,0.31


- **MAE**: error absoluto promedio en cm. Es fácil de leer.
- **RMSE**: penaliza más los errores grandes. Si es parecido al MAE, no hay errores extremos.
- **R²**: proporción de la variabilidad del largo del pétalo que explica el modelo (1 = perfecto).
- El **modelo base** predice siempre la media; sirve de referencia.

### 2.2 Coeficientes e intercepto

In [18]:
print(regression.coefficients().round(4).to_string(index=False))
print()
for name, value in regression.intercepts().items():
    print(f"{name}: {value:.4f}")

          variable  coef_estandarizado  coef_por_cm
      sepal_length              0.5574       0.6385
       sepal_width             -0.0840      -0.1856
       petal_width              0.4953       0.6429
species_versicolor              1.3461       1.3461
 species_virginica              1.8114       1.8114

intercepto_estandarizado: 2.7509
intercepto_unidades_originales: -1.2210


- `coef_estandarizado`: cambio en cm de `petal_length` por cada **desviación estándar** de la variable
  (escala del Pipeline).
- `coef_por_cm`: el mismo efecto expresado **por cada cm**. Es el coeficiente estandarizado dividido
  por la desviación estándar de entrenamiento.
- `species_*`: diferencia promedio frente a **setosa** con iguales medidas.
- El intercepto estandarizado es la predicción para una setosa con medidas promedio.

In [19]:
for line in interpret_regression(regression, reg_metrics, reg_cv, config.CV_FOLDS):
    print("-", line)

- R² = 0.970: el modelo explica el 97.0% de la variabilidad de petal_length en los datos de prueba.
- MAE = 0.220 cm: en promedio la predicción se aleja 0.22 cm del valor real (5.9% del valor medio, 3.70 cm).
- RMSE = 0.292 cm (1.33 veces el MAE): algunos errores grandes pesan más que el resto.
- Referencia: un modelo que siempre predice la media tendría MAE = 1.516 cm; la regresión reduce el error en 85.5%.
- Validación cruzada (5 pliegues sobre entrenamiento): R² = 0.975 ± 0.013; el desempeño es estable entre particiones.
- Coeficiente sepal_length: +0.639 cm de petal_length por cada cm adicional (+0.557 cm por desviación estándar), con las demás variables constantes.
- Coeficiente sepal_width: -0.186 cm de petal_length por cada cm adicional (-0.084 cm por desviación estándar), con las demás variables constantes.
- Coeficiente petal_width: +0.643 cm de petal_length por cada cm adicional (+0.495 cm por desviación estándar), con las demás variables constantes.
- Coeficiente species_ver

**Limitaciones:** las medidas están muy correlacionadas entre sí y con la especie
(multicolinealidad), así que los coeficientes individuales no deben leerse como efectos causales.
Además, el modelo necesita conocer la especie y supone relaciones lineales. El dataset es pequeño.

## 3. Modelo de clasificación

**Qué se clasifica:** la **especie** de la flor (`setosa`, `versicolor`, `virginica`) a partir de
sus cuatro medidas (`sepal_length`, `sepal_width`, `petal_length`, `petal_width`). Ninguna de
ellas revela directamente la respuesta.

**Algoritmo: regresión logística (multinomial).**
- Es un modelo lineal, sencillo e interpretable: cada clase tiene sus coeficientes.
- Funciona bien con pocas variables numéricas y clases casi separables linealmente, como en Iris.
- Entrega probabilidades por clase.

**Hiperparámetros:** `max_iter=1000` (para asegurar convergencia) y `random_state=42`. El resto son
predeterminados: `C=1.0`, `solver="lbfgs"` y regularización L2 (`l1_ratio=0.0`).
El `StandardScaler` va en el Pipeline, ajustado solo con entrenamiento.

In [20]:
distribution = df_clean["species"].value_counts().sort_index()
print(describe_class_balance(distribution))
distribution

Distribución de clases: setosa=48, versicolor=50, virginica=49. Las clases están balanceadas, así que la exactitud es representativa; además se reportan precisión, recall y F1 macro, que dan el mismo peso a cada clase.


species
setosa        48
versicolor    50
virginica     49
Name: count, dtype: int64

In [21]:
classifier = LogisticRegressionClassifier(
    config.CLASSIFICATION_FEATURES, config.CLASSIFICATION_TARGET,
    max_iter=config.LOGISTIC_MAX_ITER, random_state=config.RANDOM_STATE)
Xc_train, Xc_test, yc_train, yc_test = split_data(
    df_clean, config.CLASSIFICATION_FEATURES, config.CLASSIFICATION_TARGET,
    config.TEST_SIZE, config.RANDOM_STATE, stratify_column=config.CLASSIFICATION_TARGET)

classifier.fit(Xc_train, yc_train)

print("Entrenamiento:", Xc_train.shape, "| Prueba:", Xc_test.shape)
print("Clases por conjunto de prueba:", yc_test.value_counts().to_dict())
print("Hiperparámetros:", classifier.params())

Entrenamiento: (102, 4) | Prueba: (45, 4)
Clases por conjunto de prueba: {'setosa': 15, 'versicolor': 15, 'virginica': 15}
Hiperparámetros: {'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 1000, 'n_jobs': None, 'penalty': 'deprecated', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}


### 3.1 Evaluación: matriz de confusión y métricas

Las clases están **balanceadas**, por eso la **exactitud** es representativa. Se agregan
**precisión**, **recall** y **F1 macro**, que dan igual peso a cada especie y muestran en qué
clase se concentran los errores.

In [22]:
clf_metrics = classifier.evaluate(Xc_test, yc_test)
clf_cv = classifier.cross_validate(Xc_train, yc_train, config.CV_FOLDS, config.RANDOM_STATE)

clf_metrics["confusion_matrix"]

,pred_setosa,pred_versicolor,pred_virginica
real_setosa,15,0,0
real_versicolor,0,14,1
real_virginica,0,3,12


In [23]:
print(clf_metrics["report_text"])
print(f"Exactitud CV ({config.CV_FOLDS} pliegues): "
      f"{clf_cv['accuracy_mean']:.3f} ± {clf_cv['accuracy_std']:.3f}")

              precision    recall  f1-score   support

      setosa      1.000     1.000     1.000        15
  versicolor      0.824     0.933     0.875        15
   virginica      0.923     0.800     0.857        15

    accuracy                          0.911        45
   macro avg      0.916     0.911     0.911        45
weighted avg      0.916     0.911     0.911        45

Exactitud CV (5 pliegues): 0.980 ± 0.040


In [24]:
# Flores de prueba mal clasificadas
errors = Xc_test.copy()
errors["real"] = yc_test.astype(str)
errors["predicho"] = classifier.predict(Xc_test)
errors[errors["real"] != errors["predicho"]]

,sepal_length,sepal_width,petal_length,petal_width,real,predicho
104,4.9,2.5,4.5,1.7,virginica,versicolor
75,6.7,3.0,5.0,1.7,versicolor,virginica
131,6.3,2.8,5.1,1.5,virginica,versicolor
132,6.1,2.6,5.6,1.4,virginica,versicolor


In [25]:
classifier.coefficients().round(3)

,sepal_length,sepal_width,petal_length,petal_width
setosa,-1.090,0.927,-1.698,-1.561
versicolor,0.437,-0.439,-0.079,-0.802
virginica,0.653,-0.488,1.776,2.363


In [26]:
for line in interpret_classification(classifier, clf_metrics, clf_cv, config.CV_FOLDS):
    print("-", line)

- Exactitud = 0.911: 41 de 45 flores de prueba se clasificaron correctamente.
- Precisión macro = 0.916, recall macro = 0.911, F1 macro = 0.911.
- Clase(s) mejor reconocida(s): setosa (F1 = 1.000).
- Clase(s) con menor F1: virginica (F1 = 0.857).
- Errores de la matriz de confusión: 1 flor(es) versicolor clasificada(s) como virginica; 3 flor(es) virginica clasificada(s) como versicolor.
- Validación cruzada estratificada (5 pliegues sobre entrenamiento): exactitud = 0.980 ± 0.040.
- Para la clase setosa, la variable más influyente es petal_length (coef. -1.698 sobre datos estandarizados).
- Para la clase versicolor, la variable más influyente es petal_width (coef. -0.802 sobre datos estandarizados).
- Para la clase virginica, la variable más influyente es petal_width (coef. +2.363 sobre datos estandarizados).


**Interpretación:** `setosa` se reconoce sin errores porque sus pétalos son mucho más pequeños.
Los errores se concentran entre **versicolor y virginica**, cuyas medidas se solapan. Las flores
mal clasificadas tienen valores de pétalo en la zona de frontera entre ambas especies.

La exactitud en validación cruzada es mayor que en prueba. Con solo 45 flores de prueba, cada
error resta 2.2 puntos, así que el resultado depende de qué flores caen en prueba.

**Limitaciones:** el dataset es pequeño, el modelo es lineal y tiene una frontera difícil entre
versicolor y virginica. Además, Iris es un conjunto "de laboratorio" limpio.

## 4. Resumen automático `.txt`

Se ejecuta el flujo completo `run_experiment()`, el mismo que corre `main.py` en GitHub Actions.
Se comprueba que sus métricas coinciden con las obtenidas paso a paso arriba y se genera el resumen.

In [27]:
results = run_experiment()

assert np.isclose(results.regression_metrics["r2"], reg_metrics["r2"])
assert np.isclose(results.regression_metrics["mae"], reg_metrics["mae"])
assert np.isclose(results.classification_metrics["accuracy"], clf_metrics["accuracy"])
assert results.classification_metrics["confusion_matrix"].equals(clf_metrics["confusion_matrix"])
print("Las métricas del flujo automático coinciden con las del notebook.")

OUTPUT_PATH = PROJECT_ROOT / "output" / config.SUMMARY_FILENAME
saved_path = ModelSummaryWriter(results).save(OUTPUT_PATH)
print("Resumen guardado en:", saved_path)
print("¿Existe?", saved_path.exists(), "| tamaño:", saved_path.stat().st_size, "bytes")

Las métricas del flujo automático coinciden con las del notebook.
Resumen guardado en: C:\Proyectos\MachineLearningIU\S30_Unidad1_EA2_Modelos_de_Prediccion_y_Clasificacion\output\resumen_modelos.txt
¿Existe? True | tamaño: 10124 bytes


In [28]:
print(saved_path.read_text(encoding="utf-8"))

S30 - EA2 UNIDAD 1 - MODELOS DE PREDICCIÓN Y CLASIFICACIÓN
Resumen generado automáticamente por código
Fecha de ejecución: 2026-10-25 22:05:32 (UTC-0500)
Entorno: Python 3.12.10 | pandas 3.1.0 | scikit-learn 1.9.1 | datasets 5.1.0

1. DATASET
Identificador: scikit-learn/iris
Enlace:        https://huggingface.co/datasets/scikit-learn/iris
Split:         train (único split disponible)
Obtención:     automática con datasets.load_dataset() -> pandas
Tamaño original: 150 filas x 6 columnas
Columnas originales: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']
Tamaño limpio:   147 filas x 5 columnas
Columnas limpias:    ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

2. PREPROCESAMIENTO
Operaciones sobre el DataFrame completo (no aprenden de los datos):
  1. Renombrar columnas
     Motivo: Nombres en snake_case, sin la unidad 'Cm'.
     Efecto: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species'] -> 

## 5. Ejecución automática en GitHub Actions

El workflow `.github/workflows/s30_modelos_ml.yml` (en la raíz del repositorio) se activa con cada
**push** a `main` que modifique esta carpeta. Hace lo siguiente:

1. Instala Python 3.12 y `requirements.txt`.
2. Ejecuta `python main.py`: descarga el dataset desde Hugging Face, lo preprocesa, entrena y
   evalúa los dos modelos y genera `output/resumen_modelos.txt`.
3. Publica el resumen como **artefacto descargable** `resumen-modelos`.